# Chapter 9 &mdash; The GNFA: Adding `Real_I` and `Real_F` to Stand On

**Concept 1 of the Chapter 9 decomposition:** *The GNFA: Adding `Real_I` and `Real_F` to Stand On*

Wrap the NFA in one new initial and one new final state joined by $\varepsilon$ edges, so elimination has somewhere to stand.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter9/Concept-The-GNFA/Concept-The-GNFA.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.Def_NFA2RE     import *
from jove.AnimateNFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


To turn an NFA into a regular expression we delete states one at a time until only two
remain, and read the label between them. For that to work we need **exactly one**
initial and **exactly one** final state, and neither may be deleted.

So wrap the machine: add **`Real_I`** with an $\varepsilon$ edge to every original
start state, and **`Real_F`** with an $\varepsilon$ edge from every original final
state. The result is a **GNFA** &mdash; a *generalized* NFA whose edges carry **regular
expressions**, not just symbols.

`Real_I` has no incoming edges and `Real_F` no outgoing ones, which is exactly what
makes them safe to stand on.

## 2. Definitions

### An NFA, and its GNFA wrapper

In [ ]:
N = md2mc('''NFA
I : 0 -> I
I : 1 -> F
F : 0 -> F
F : 1 -> I
''')
Gw = mk_gnfa(N)
print("NFA  : Q0 = %s, F = %s" % (sorted(N["Q0"]), sorted(N["F"])))
print("GNFA : Q0 = %s, F = %s" % (sorted(Gw["Q0"]), sorted(Gw["F"])))
print("GNFA states :", sorted(Gw["Q"]))

### A machine with several start and several final states

In [ ]:
Multi = md2mc('''NFA
I1 : 0 -> F1
I2 : 1 -> F2
F1 : 0 -> F1
F2 : 1 -> F2
''')
GM = mk_gnfa(Multi)

## 3. Tests

The wrapper adds exactly two states.

In [ ]:
print("|Q| : NFA %d -> GNFA %d" % (len(N["Q"]), len(Gw["Q"])))
assert len(Gw["Q"]) == len(N["Q"]) + 2
assert Gw["Q0"] == {"Real_I"} and Gw["F"] == {"Real_F"}

`Real_I` has no incoming edges; `Real_F` has none outgoing. That is why they survive.

In [ ]:
into_I  = [(p, q) for (p, _), qs in Gw["Delta"].items() for q in qs if q == "Real_I"]
outof_F = [k for k in Gw["Delta"] if k[0] == "Real_F"]
print("edges into Real_I   :", into_I)
print("edges out of Real_F :", outof_F)
assert not into_I and not outof_F

Several start states collapse into one, via $\varepsilon$ edges.

In [ ]:
print("Multi : Q0 = %s, F = %s" % (sorted(Multi["Q0"]), sorted(Multi["F"])))
print("GNFA  : Q0 = %s, F = %s" % (sorted(GM["Q0"]), sorted(GM["F"])))
assert len(GM["Q0"]) == 1 and len(GM["F"]) == 1
print("\nedges from Real_I :", sorted(Edges_Exist_Via(GM, "Real_I", q) and q
                                      for q in sorted(GM["Q"]) if Edges_Exist_Via(GM, "Real_I", q)))

Wrapping does not change the language &mdash; the added edges are $\varepsilon$.

In [ ]:
_, _, restr = del_gnfa_states(mk_gnfa(N))
print("RE from the GNFA :", restr)
from itertools import product
D1 = min_dfa(nfa2dfa(N))
D2 = min_dfa(nfa2dfa(re2nfa(restr)))
print("same language as the original NFA? ", iso_dfa(D1, D2))
assert iso_dfa(D1, D2)

GNFA edges carry **regular expressions**; the original ones carry symbols.

In [ ]:
print("some GNFA edge labels :")
for k, v in list(sorted(Gw["Delta"].items()))[:6]:
    print("   %-22s -> %s" % (k, sorted(v)))

## 4. Animation

The machine before wrapping.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateNFA import *
AnimateNFA(N, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Wrap a machine whose start state is also final. Where do the $\varepsilon$ edges go?
2. Why can `Real_I` never be chosen for deletion?
3. What would go wrong if you reused an existing state as `Real_I`?

In [ ]:
# Your work for the exercises above.